<img src = "../assets/bbai-banner.png" align="center" height = "25%">

# RUL Model Predictions from MLFlow #

This notebook will demonstrate using saved models from an MLFlow instance and using them to make predictions using the NASA CMAPSS dataset.  The results will be demonstrated in graphs that can be shared with stakeholders.

In [ ]:
import mlflow
import os
import sys
import pandas as pd
from mlflow.models import infer_signature
import ast
import numpy as np
from matplotlib import pyplot as plt

module_paths = [
    os.path.abspath(os.path.join('..')),
]
for module_path in module_paths:
    if module_path not in sys.path:
        sys.path.append(module_path)

from pdm.rul_model import RULModel
from pdm.config import RULConfig

In [ ]:

mlflow.set_tracking_uri(uri="http://127.0.0.1:8080")

# This is pulled from the "best" run in MLFlow
# logged_model_run_id = '2e4f3c18ad19488994ee5fede05bdb52'  # silent-gnat-31
# logged_model_run_id = 'f35088f5219245d0a1fc73bcbab9789f'  # funny-mare-148
# logged_model_run_id = '5e7bf0ca38834955a632b7f350317747'  # nimble_vole_309
logged_model_run_id = '027d4f62ba3349f1a5211f8cb6374d16' # valuable-dove-872
# logged_model_run_id = 'ce365d7afb5b41e3b4b45ee3295d1cf2' # treasured-auk-336
logged_model = 'runs:/'+logged_model_run_id+'/model'

# Load model as a PyFuncModel.
# Loading from MLFlow is causing an error.  
#loaded_model = mlflow.pyfunc.load_model(logged_model)

# We can retrieve the parameters of the run using the MLflow API
mlflow_server_uri="http://127.0.0.1:8080"
mlflow_client = mlflow.tracking.MlflowClient(
    tracking_uri=mlflow_server_uri
)

run_data_dict = mlflow_client.get_run(logged_model_run_id).data.to_dictionary()

# Run Dictionary contains params
config_params = run_data_dict['params']
config_params['sensor_columns'] = ast.literal_eval(config_params['sensor_columns'])
print(config_params)

# Run dictionary also contains metrics
# print(run_data_dict['metrics'])

In [ ]:
# Identify the dataset we are working with
filename = "FD001"
filepath = '../data/training/train_'+filename+'.txt'

# Identify the Column Names for the data
column_names = ["id","cycle","setting1","setting2", "setting3",
        "s1","s2","s3","s4","s5","s6","s7","s8","s9","s10","s11","s12","s13","s14","s15","s16","s17","s18","s19","s20","s21",]

# Load the file into Pandas dataframe
test_data = pd.read_csv(filepath, sep=" ", header=None)

test_data.columns = column_names



In [ ]:
#  Instantiate a rul_model (default config works)
config = RULConfig(**config_params)
rul_model = RULModel(**config.model_dump())
print("Opening Model from: "+ config_params['save_file_name'])
rul_model.load_model_from_disk()



In [ ]:
def engine_prediction(true_rul, test_data):
    num_rows = test_data.shape[0]
    num_columns = test_data.shape[1]
    seq_len = rul_model.sequence_length

    true_rul_list = []
    for cycle in range(0,num_rows):
        true_rul_list.append(min(true_rul+num_rows-cycle, rul_model.max_RUL))

    predictions_list = []
    for index in range(0,(num_rows-seq_len+1)):
        subset_df = test_data.iloc[index:index+seq_len, 0:num_columns]
        print(subset_df.shape)
        print(subset_df)
        predictions = rul_model.batch_predict(subset_df)
        if not predictions_list:
            for cycle in range(0, (seq_len-1)):
                predictions_list.append(min(predictions.at[0] + seq_len - cycle, rul_model.max_RUL))
        predictions_list.append(predictions.at[0])

    print(predictions_list)

    print(len(predictions_list))
    results_frame = pd.DataFrame({'True_RUL':true_rul_list, 'Predicted_RUL':predictions_list})
    return results_frame

In [ ]:
#plot the prediction vs the ground truth
def plot_batch_result(y_true, y_pred, title=None):

    """ A function to plot the predicted RUL vs the target RUL for all engines"""
    zero_data = [0] * len(y_true)
    y_zero = pd.Series(zero_data)

    fig = plt.figure(figsize = (16, 12))
    plt.grid(True,lw=0.1)
    plt.rcParams['axes.linewidth'] = 0.4  # set the value globally
    plt.plot(y_true, 'g', linewidth=0.5)
    plt.plot(y_pred, 'k', linewidth=0.5)
    plt.plot(y_zero, 'r', linewidth=0.5,linestyle='--')
    plt.title(title,fontsize=8)
    plt.xlabel('Cycle', fontsize=12)
    plt.xticks(fontsize=5)
    plt.ylabel('RUL', fontsize=12)
    plt.yticks(fontsize=5)
    plt.legend(['Actual RUL', 'Predicted RUL', 'Maintenance Point'],  loc='upper right', fontsize=5)
    plt.show()
    fig.savefig("../results/"  + title + '.tiff', dpi=90, format='tiff', bbox_inches='tight')
    return fig

In [ ]:
truth_file = '../data/RUL/RUL_train_'+filename+'.txt'
truth_data = pd.read_csv(truth_file, sep=" ", header=None)
truth_data.columns=['true_rul']
num_engines = truth_data.shape[0]
for engine in range(0,num_engines):
    condition = test_data['id']==(engine+1)
    results = engine_prediction(truth_data.at[engine,'true_rul'], test_data[condition])
    plot_batch_result(results['True_RUL'], results['Predicted_RUL'], "Engine "+str(engine+1)+" of Test Dataset "+ filename)

In [ ]:
# Read the file and create a pandas Series
# truth_file_path = "../data/RUL/RUL_FD001.txt"
# truth_data = pd.read_csv(truth_file_path, header=None)
# truth_data.columns=['trueRUL']

# # Flatten the predictions results
# #predictions = pd.Series(predictions.flatten(), name="predictions")

# evaluation_df = pd.merge(predictions, truth_data, left_index=True, right_index=True)

# evaluation_df = evaluation_df.sort_values('trueRUL', ascending=False).reset_index(drop=True)

# plot_batch_result(evaluation_df['trueRUL'], evaluation_df['predictions'], "Results")

#plot_batch_result(results_frame['True_RUL'], results_frame['Predicted_RUL'], "Engine 5 of FD001")